In [2]:
!pip install -q transformers datasets torch accelerate sentencepiece

In [3]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import os
import re
from typing import List, Dict, Tuple, Optional
import warnings
import shutil
from difflib import SequenceMatcher
warnings.filterwarnings('ignore')

2026-02-09 08:46:48.883608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770626809.070027      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770626809.121350      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770626809.562083      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770626809.562129      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770626809.562132      55 computation_placer.cc:177] computation placer alr

In [4]:
class Config:
    TRAIN_ZIP = '/kaggle/input/dimaste/subtask2'
    DATA_DIR = './subtask2_data'
    OUTPUT_DIR = './improved_outputs'
    CHECKPOINT_DIR = './improved_checkpoints'
    
    MODEL_NAME = 't5-base'
    MAX_INPUT_LENGTH = 128
    MAX_OUTPUT_LENGTH = 256
    
    BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2
    EPOCHS = 15
    LEARNING_RATE = 5e-5
    WARMUP_RATIO = 0.15
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    LABEL_SMOOTHING = 0.1
    EARLY_STOP_PATIENCE = 4
    
    VA_WEIGHT_SCHEDULE = {'start': 0.3, 'middle': 0.5, 'end': 0.7}
    VA_DIVERSITY_WEIGHT = 0.1
    VA_MIN = 1.0
    VA_MAX = 9.0
    NUM_BEAMS = 5
    SPAN_MATCH_THRESHOLD = 0.75
    
    MIN_SENTIMENT_WORDS = ['good', 'bad', 'great', 'excellent', 'poor', 'terrible', 
                           'amazing', 'awful', 'nice', 'horrible', 'love', 'hate',
                           'best', 'worst', 'disappointing', 'fantastic', 'slow',
                           'fast', 'clean', 'dirty', 'delicious', 'bland']
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED = 42

config = Config()

os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

files_to_copy = [
    "eng_laptop_dev_task2.jsonl",
    "eng_laptop_train_alltasks.jsonl",
    "eng_laptop_test_task2.jsonl",
    "eng_restaurant_dev_task2.jsonl",
    "eng_restaurant_test_task2.jsonl",
    "eng_restaurant_train_alltasks.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"\nDevice: {config.DEVICE}")
print(f"Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")

Copied eng_laptop_dev_task2.jsonl
Copied eng_laptop_train_alltasks.jsonl
Copied eng_laptop_test_task2.jsonl
Copied eng_restaurant_dev_task2.jsonl
Copied eng_restaurant_test_task2.jsonl
Copied eng_restaurant_train_alltasks.jsonl

Device: cuda
Effective batch size: 32


## IMPROVEMENT #1: Span Matcher (Eliminates Hallucinations)

In [5]:
class SpanMatcher:
    @staticmethod
    def similarity_ratio(s1: str, s2: str) -> float:
        return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()
    
    @staticmethod
    def find_best_match(text: str, predicted_span: str, threshold: float = 0.75) -> Optional[str]:
        text_lower = text.lower()
        pred_lower = predicted_span.lower()
        
        if pred_lower in text_lower:
            start = text_lower.index(pred_lower)
            return text[start:start + len(predicted_span)]
        
        pred_clean = re.sub(r'[^\w\s]', '', pred_lower)
        text_clean = re.sub(r'[^\w\s]', '', text_lower)
        if pred_clean in text_clean:
            start = text_clean.index(pred_clean)
            return text[start:start + len(pred_clean)]
        
        pred_words = predicted_span.split()
        if len(pred_words) == 1:
            words_in_text = text.split()
            best_match = None
            best_score = 0.0
            
            for word in words_in_text:
                score = SpanMatcher.similarity_ratio(predicted_span, word)
                if score > best_score and score >= threshold:
                    best_score = score
                    best_match = word
            
            return best_match
        else:
            window_size = len(pred_words)
            words_in_text = text.split()
            best_match = None
            best_score = 0.0
            
            for i in range(len(words_in_text) - window_size + 1):
                window = ' '.join(words_in_text[i:i+window_size])
                score = SpanMatcher.similarity_ratio(predicted_span, window)
                if score > best_score and score >= threshold:
                    best_score = score
                    best_match = window
            
            return best_match
        
        return None

print("SpanMatcher ready - eliminates hallucinations")

SpanMatcher ready - eliminates hallucinations


## IMPROVEMENT #2: Sentiment Detector (Reduces Empty Predictions)

In [6]:
class SentimentDetector:
    @staticmethod
    def contains_sentiment(text: str) -> bool:
        text_lower = text.lower()
        for word in config.MIN_SENTIMENT_WORDS:
            if word in text_lower:
                return True
        return False
    
    @staticmethod
    def extract_fallback_triplet(text: str) -> Optional[Dict]:
        text_lower = text.lower()
        words = text.split()
        
        opinion = None
        opinion_idx = -1
        for i, word in enumerate(words):
            if word.lower() in config.MIN_SENTIMENT_WORDS:
                opinion = word
                opinion_idx = i
                break
        
        if opinion is None:
            return None
        
        aspect = None
        for i in range(max(0, opinion_idx - 3), opinion_idx):
            word = words[i]
            if len(word) > 2 and word.lower() not in ['the', 'a', 'an', 'is', 'was', 'were']:
                aspect = word
        
        if aspect is None:
            aspect = 'it'
        
        if opinion.lower() in ['good', 'great', 'excellent', 'amazing', 'fantastic', 'love', 'best', 'delicious', 'clean', 'fast', 'nice']:
            va = "7.00#6.50"
        elif opinion.lower() in ['bad', 'poor', 'terrible', 'awful', 'horrible', 'hate', 'worst', 'disappointing', 'bland', 'dirty', 'slow']:
            va = "3.50#6.00"
        else:
            va = "5.00#5.00"
        
        return {'Aspect': aspect, 'Opinion': opinion, 'VA': va}

print("SentimentDetector ready - reduces empty predictions")

SentimentDetector ready - reduces empty predictions


## IMPROVEMENT #3: Enhanced VA Regression Head

In [7]:
class VAHead(nn.Module):
    def __init__(self, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        
        self.valence_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        self.arousal_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, hidden_states):
        pooled = hidden_states.mean(dim=1)
        valence = self.valence_head(pooled)
        arousal = self.arousal_head(pooled)
        return valence, arousal

print("Enhanced VAHead ready - separate heads for V and A")

Enhanced VAHead ready - separate heads for V and A


In [8]:
def normalize_va(v: float, a: float) -> Tuple[float, float]:
    v_norm = (v - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)
    a_norm = (a - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)
    return v_norm, a_norm

def denormalize_va(v_norm: float, a_norm: float) -> Tuple[float, float]:
    v = v_norm * (config.VA_MAX - config.VA_MIN) + config.VA_MIN
    a = a_norm * (config.VA_MAX - config.VA_MIN) + config.VA_MIN
    v = max(config.VA_MIN, min(config.VA_MAX, v))
    a = max(config.VA_MIN, min(config.VA_MAX, a))
    return round(v, 2), round(a, 2)

def format_va(v: float, a: float) -> str:
    return f"{v:.2f}#{a:.2f}"

print("VA normalization functions ready")

VA normalization functions ready


In [9]:
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    null_filtered = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            
            if 'Quadruplet' in item:
                item['Triplet'] = []
                for quad in item['Quadruplet']:
                    if quad['Aspect'] != 'NULL' and quad['Opinion'] != 'NULL':
                        item['Triplet'].append({
                            'Aspect': quad['Aspect'],
                            'Opinion': quad['Opinion'],
                            'VA': quad['VA']
                        })
                    else:
                        null_filtered += 1
            
            data.append(item)
    
    total_triplets = sum(len(item.get('Triplet', [])) for item in data)
    print(f"Loaded {len(data)} samples from {os.path.basename(file_path)}")
    print(f"  Total triplets: {total_triplets}")
    if null_filtered > 0:
        print(f"  Filtered NULLs: {null_filtered}")
    
    return data

print("Data loading function ready")

Data loading function ready


In [10]:
class DimASTEDataset(Dataset):
    def __init__(self, data: List[Dict], tokenizer, max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_triplets = 10
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        input_text = f"extract triplets: {item['Text']}"
        
        triplets = item.get('Triplet', [])
        if len(triplets) == 0:
            target_text = "[NONE]"
            va_values = []
        else:
            target_parts = []
            va_values = []
            for trip in triplets:
                target_parts.append(f"{trip['Aspect']} | {trip['Opinion']}")
                v, a = map(float, trip['VA'].split('#'))
                v_norm, a_norm = normalize_va(v, a)
                va_values.append([v_norm, a_norm])
            target_text = " [SEP] ".join(target_parts)
        
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            target_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        if va_values:
            va_tensor = torch.tensor(va_values, dtype=torch.float32)
            if len(va_values) < self.max_triplets:
                padding = torch.zeros((self.max_triplets - len(va_values), 2))
                va_tensor = torch.cat([va_tensor, padding], dim=0)
        else:
            va_tensor = torch.zeros((self.max_triplets, 2))
        
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze(),
            'va_values': va_tensor,
            'num_triplets': len(triplets),
            'text': item['Text']
        }

class DimASTETestDataset(Dataset):
    def __init__(self, data: List[Dict], tokenizer, max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        input_text = f"extract triplets: {item['Text']}"
        
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'text': item['Text']
        }

print("Dataset classes ready - FIXED tensor size")

Dataset classes ready - FIXED tensor size


In [11]:
def parse_generation_output(generated_text: str, original_text: str) -> List[Dict]:
    triplets = []
    
    if generated_text.strip() == "[NONE]" or not generated_text.strip():
        return triplets
    
    segments = generated_text.split('[SEP]')
    matcher = SpanMatcher()
    
    for segment in segments:
        segment = segment.strip()
        if not segment or '|' not in segment:
            continue
        
        parts = segment.split('|')
        if len(parts) != 2:
            continue
        
        predicted_aspect = parts[0].strip()
        predicted_opinion = parts[1].strip()
        
        matched_aspect = matcher.find_best_match(
            original_text, 
            predicted_aspect, 
            config.SPAN_MATCH_THRESHOLD
        )
        
        matched_opinion = matcher.find_best_match(
            original_text, 
            predicted_opinion, 
            config.SPAN_MATCH_THRESHOLD
        )
        
        if matched_aspect and matched_opinion:
            triplets.append({
                'Aspect': matched_aspect,
                'Opinion': matched_opinion
            })
    
    return triplets

print("Output parsing with span matching ready")

Output parsing with span matching ready


## IMPROVEMENT #4: VA Diversity Loss

In [12]:
def compute_va_diversity_loss(va_predictions: torch.Tensor) -> torch.Tensor:
    if va_predictions.size(0) < 2:
        return torch.tensor(0.0, device=va_predictions.device)
    
    v_var = torch.var(va_predictions[:, 0])
    a_var = torch.var(va_predictions[:, 1])
    diversity_loss = -(v_var + a_var)
    
    return diversity_loss

print("VA diversity loss ready - prevents collapse")

VA diversity loss ready - prevents collapse


## IMPROVEMENT #5: Progressive VA Weighting

In [13]:
def get_va_weight(epoch: int, total_epochs: int) -> float:
    progress = epoch / total_epochs
    
    if progress < 0.3:
        return config.VA_WEIGHT_SCHEDULE['start']
    elif progress < 0.7:
        return config.VA_WEIGHT_SCHEDULE['middle']
    else:
        return config.VA_WEIGHT_SCHEDULE['end']

print("Progressive VA weighting ready")
print("  Epochs 1-5: VA weight = 0.3 (focus extraction)")
print("  Epochs 6-10: VA weight = 0.5 (balanced)")
print("  Epochs 11-15: VA weight = 0.7 (refine VA)")

Progressive VA weighting ready
  Epochs 1-5: VA weight = 0.3 (focus extraction)
  Epochs 6-10: VA weight = 0.5 (balanced)
  Epochs 11-15: VA weight = 0.7 (refine VA)


In [14]:
def train_epoch(model, va_head, dataloader, optimizer, scheduler, device, epoch, total_epochs):
    model.train()
    va_head.train()
    
    total_loss = 0
    total_gen_loss = 0
    total_va_loss = 0
    total_div_loss = 0
    
    va_weight = get_va_weight(epoch, total_epochs)
    
    pbar = tqdm(dataloader, desc=f"Training (VA weight={va_weight:.2f})")
    
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        va_values = batch['va_values'].to(device)
        num_triplets = batch['num_triplets']
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )
        
        gen_loss = outputs.loss
        
        encoder_hidden = outputs.encoder_last_hidden_state
        v_pred, a_pred = va_head(encoder_hidden)
        
        va_loss = torch.tensor(0.0, device=device)
        diversity_loss = torch.tensor(0.0, device=device)
        
        valid_va_count = 0
        all_va_preds = []
        
        for i, n_trip in enumerate(num_triplets):
            if n_trip > 0:
                target_v = va_values[i][:n_trip, 0].mean()
                target_a = va_values[i][:n_trip, 1].mean()
                
                va_loss += F.mse_loss(v_pred[i], target_v)
                va_loss += F.mse_loss(a_pred[i], target_a)
                
                valid_va_count += 1
                all_va_preds.append(torch.stack([v_pred[i], a_pred[i]]))
        
        if valid_va_count > 0:
            va_loss /= valid_va_count
            
            if len(all_va_preds) > 1:
                va_tensor = torch.stack(all_va_preds).squeeze()
                diversity_loss = compute_va_diversity_loss(va_tensor)
        
        loss = (1 - va_weight) * gen_loss + va_weight * va_loss
        loss += config.VA_DIVERSITY_WEIGHT * diversity_loss
        
        loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        
        if (step + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(va_head.parameters()),
                config.GRADIENT_CLIP
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        total_gen_loss += gen_loss.item()
        total_va_loss += va_loss.item()
        total_div_loss += diversity_loss.item()
        
        pbar.set_postfix({
            'loss': f'{total_loss/(step+1):.4f}',
            'gen': f'{total_gen_loss/(step+1):.4f}',
            'va': f'{total_va_loss/(step+1):.4f}'
        })
    
    return {
        'loss': total_loss / len(dataloader),
        'gen_loss': total_gen_loss / len(dataloader),
        'va_loss': total_va_loss / len(dataloader),
        'div_loss': total_div_loss / len(dataloader)
    }

print("Training function ready")

Training function ready


In [35]:
def continuous_f1_score_triplets(predictions, references):
    """
    OFFICIAL Continuous F1 for DimASTE (Subtask 2).
    Triplets: (Aspect, Opinion, VA)
    """
    
    # Helper function to parse VA string
    def parse_va(va_string):
        """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
        v, a = va_string.split('#')
        return float(v), float(a)
    
    def calculate_va_penalty(pred_va, gold_va):
        """Official formula: dist = VA_error / D_max"""
        pred_v, pred_a = parse_va(pred_va)
        gold_v, gold_a = parse_va(gold_va)
        
        va_error = np.sqrt((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
        D_max = np.sqrt(128)  # Official normalization
        dist = va_error / D_max
        
        return 1.0 - dist
    
    def triplet_matches(pred, gold):
        """Check if triplets match on (Aspect, Opinion)"""
        # Aspect match (exact or substring)
        aspect_match = (
            pred['Aspect'] == gold['Aspect'] or
            pred['Aspect'] in gold['Aspect'] or
            gold['Aspect'] in pred['Aspect']
        )
        if not aspect_match:
            return False
        
        # Opinion match (exact or substring)
        opinion_match = (
            pred['Opinion'] == gold['Opinion'] or
            pred['Opinion'] in gold['Opinion'] or
            gold['Opinion'] in pred['Opinion']
        )
        if not opinion_match:
            return False
        
        return True
    
    # Calculate scores for predictions (for precision)
    pred_scores = []
    for pred in predictions:
        best_score = 0.0
        for gold in references:
            if triplet_matches(pred, gold):
                va_score = calculate_va_penalty(pred['VA'], gold['VA'])
                best_score = max(best_score, va_score)
        pred_scores.append(best_score)
    
    # Calculate scores for references (for recall)
    ref_scores = []
    for gold in references:
        best_score = 0.0
        for pred in predictions:
            if triplet_matches(pred, gold):
                va_score = calculate_va_penalty(pred['VA'], gold['VA'])
                best_score = max(best_score, va_score)
        ref_scores.append(best_score)
    
    # Compute continuous precision and recall
    cPrecision = sum(pred_scores) / len(predictions) if predictions else 0.0
    cRecall = sum(ref_scores) / len(references) if references else 0.0
    
    # Compute continuous F1
    cF1 = 2 * (cPrecision * cRecall) / (cPrecision + cRecall) if (cPrecision + cRecall) > 0 else 0.0
    
    # Calculate VA RMSE
    v_errors = []
    a_errors = []
    va_values = []
    
    for pred in predictions:
        for gold in references:
            if triplet_matches(pred, gold):
                pred_v, pred_a = parse_va(pred['VA'])
                gold_v, gold_a = parse_va(gold['VA'])
                
                v_errors.append((pred_v - gold_v) ** 2)
                a_errors.append((pred_a - gold_a) ** 2)
                va_values.append([pred_v, pred_a])
    
    va_rmse_v = np.sqrt(np.mean(v_errors)) if v_errors else float('inf')
    va_rmse_a = np.sqrt(np.mean(a_errors)) if a_errors else float('inf')
    va_variance = np.var(va_values) if va_values else 0.0
    
    return {
        'cF1': cF1,
        'cPrecision': cPrecision,
        'cRecall': cRecall,
        'va_rmse_v': va_rmse_v,
        'va_rmse_a': va_rmse_a,
        'va_variance': va_variance,
        'num_predictions': len(predictions),
        'num_references': len(references)
    }



In [36]:
def predict_with_va(model, va_head, dataloader, tokenizer, device):
    model.eval()
    va_head.eval()
    
    all_predictions = []
    detector = SentimentDetector()
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            texts = batch['text']
            
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            encoder_outputs = model.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
            
            v_pred, a_pred = va_head(encoder_outputs.last_hidden_state)
            
            for i in range(len(generated_ids)):
                generated_text = tokenizer.decode(generated_ids[i], skip_special_tokens=True)
                text = texts[i]
                
                triplets = parse_generation_output(generated_text, text)
                
                if len(triplets) == 0 and detector.contains_sentiment(text):
                    fallback = detector.extract_fallback_triplet(text)
                    if fallback:
                        triplets = [fallback]
                
                v_denorm, a_denorm = denormalize_va(v_pred[i].item(), a_pred[i].item())
                va_str = format_va(v_denorm, a_denorm)
                
                for trip in triplets:
                    if 'VA' not in trip:
                        trip['VA'] = va_str
                
                all_predictions.append(triplets)
    
    return all_predictions

def save_predictions(predictions: List[List[Dict]], original_data: List[Dict], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for i, (preds, item) in enumerate(zip(predictions, original_data)):
            output = {
                'ID': item['ID'],
                'Text': item['Text'],
                'Triplet': preds
            }
            f.write(json.dumps(output, ensure_ascii=False) + '\n')
    
    print(f"Saved: {output_path}")

print("Prediction functions ready")

Prediction functions ready


In [ ]:
def evaluate(model, va_head, dataloader, tokenizer, device):
    """
    Evaluate using CONTINUOUS F1 metric.
    """
    model.eval()
    va_head.eval()
    
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            va_values = batch['va_values'].to(device)
            num_triplets = batch['num_triplets']
            texts = batch['text']
            
            # Generation loss
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=True
            )
            
            total_loss += outputs.loss.item()
            
            # Generate predictions
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            # Get VA predictions
            encoder_hidden = outputs.encoder_last_hidden_state
            v_pred, a_pred = va_head(encoder_hidden)
            
            # Process each sample
            for i in range(len(generated_ids)):
                generated_text = tokenizer.decode(generated_ids[i], skip_special_tokens=True)
                target_text = tokenizer.decode(labels[i], skip_special_tokens=True)
                
                pred_triplets = parse_generation_output(generated_text, texts[i])
                target_triplets = parse_generation_output(target_text, texts[i])
                
                # Add predicted VA to prediction triplets
                if num_triplets[i] > 0:
                    pred_v_denorm, pred_a_denorm = denormalize_va(
                        v_pred[i].item(), 
                        a_pred[i].item()
                    )
                    va_str = f"{pred_v_denorm:.2f}#{pred_a_denorm:.2f}"
                    
                    for triplet in pred_triplets:
                        triplet['VA'] = va_str
                
                # Add gold VA to target triplets from batch
                if num_triplets[i] > 0:
                    gold_va_tensor = va_values[i][:num_triplets[i]]
                    
                    for j, triplet in enumerate(target_triplets):
                        if j < num_triplets[i]:
                            target_v_denorm, target_a_denorm = denormalize_va(
                                gold_va_tensor[j, 0].item(),
                                gold_va_tensor[j, 1].item()
                            )
                            triplet['VA'] = f"{target_v_denorm:.2f}#{target_a_denorm:.2f}"
                        else:
                            # Use average if more triplets than VA values
                            mean_v = gold_va_tensor[:, 0].mean().item()
                            mean_a = gold_va_tensor[:, 1].mean().item()
                            mean_v_denorm, mean_a_denorm = denormalize_va(mean_v, mean_a)
                            triplet['VA'] = f"{mean_v_denorm:.2f}#{mean_a_denorm:.2f}"
                
                all_predictions.append(pred_triplets)
                all_targets.append(target_triplets)
    
    # Flatten predictions and targets
    flat_preds = [t for triplets in all_predictions for t in triplets if 'VA' in t]
    flat_targets = [t for triplets in all_targets for t in triplets if 'VA' in t]
    
    # Calculate CONTINUOUS F1 (THIS IS THE KEY LINE!)
    metrics = continuous_f1_score_triplets(flat_preds, flat_targets)
    metrics['loss'] = total_loss / len(dataloader)
    
    return metrics

print("Evaluation function loaded - uses CONTINUOUS F1")

## Load Data

In [37]:
print("Loading data...")
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_data = train_laptop + train_rest

dev_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task2.jsonl')
dev_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task2.jsonl')
dev_data = dev_laptop + dev_rest

print(f"\nTotal train: {len(train_data)}")
print(f"Total dev: {len(dev_data)}")

Loading data...
Loaded 4076 samples from eng_laptop_train_alltasks.jsonl
  Total triplets: 3277
  Filtered NULLs: 2496
Loaded 2284 samples from eng_restaurant_train_alltasks.jsonl
  Total triplets: 2430
  Filtered NULLs: 1229
Loaded 200 samples from eng_laptop_dev_task2.jsonl
  Total triplets: 317
Loaded 200 samples from eng_restaurant_dev_task2.jsonl
  Total triplets: 408

Total train: 6360
Total dev: 400


## Initialize Model

In [38]:
print("Initializing model...")
tokenizer = T5Tokenizer.from_pretrained(config.MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME).to(config.DEVICE)
va_head = VAHead(model.config.d_model).to(config.DEVICE)

train_dataset = DimASTEDataset(train_data, tokenizer, config.MAX_INPUT_LENGTH)
dev_dataset = DimASTEDataset(dev_data, tokenizer, config.MAX_INPUT_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=config.BATCH_SIZE)

optimizer = AdamW(
    list(model.parameters()) + list(va_head.parameters()),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader) * config.EPOCHS // config.GRADIENT_ACCUMULATION_STEPS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * config.WARMUP_RATIO),
    num_training_steps=total_steps
)

print("Model initialized")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Initializing model...
Model initialized
Total parameters: 222.9M


## Training Loop

In [39]:
print("="*80)
print("TRAINING")
print("="*80)

best_loss = float('inf')
best_f1 = 0
patience_counter = 0

for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
    print("="*80)
    
    train_metrics = train_epoch(
        model, va_head, train_loader, optimizer, scheduler, 
        config.DEVICE, epoch, config.EPOCHS
    )
    
    dev_metrics = evaluate(model, va_head, dev_loader, tokenizer, config.DEVICE)
    
    print(f"\n Results:")
    print(f"  Train Loss: {train_metrics['loss']:.4f}")
    print(f"  Dev Loss:   {dev_metrics['loss']:.4f}")
    print(f"  Precision:  {dev_metrics['cPrecision']:.4f}")
    print(f"  Recall:     {dev_metrics['cRecall']:.4f}")
    print(f"  F1 Score:   {dev_metrics['cF1']:.4f}")
    print(f"  VA RMSE (V): {dev_metrics['va_rmse_v']:.4f}")
    print(f"  VA RMSE (A): {dev_metrics['va_rmse_a']:.4f}")
    print(f"  VA Variance: {dev_metrics['va_variance']:.4f}")
    
    if dev_metrics['cF1'] > best_f1:
        best_f1 = dev_metrics['cF1']
        best_loss = dev_metrics['loss']
        patience_counter = 0
        
        model.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        tokenizer.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        torch.save(va_head.state_dict(), f'{config.CHECKPOINT_DIR}/best_va_head.pt')
        
        print(f"  ✓ Saved best model (F1: {best_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement for {patience_counter} epoch(s)")
    
    if patience_counter >= config.EARLY_STOP_PATIENCE:
        print(f"\n  Early stopping at epoch {epoch+1}")
        print(f"  Best F1: {best_f1:.4f}")
        break

print("\n" + "="*80)
print("TRAINING COMPLETED")
print("="*80)
print(f"Best F1 score: {best_f1:.4f}")

TRAINING

Epoch 1/15


Training (VA weight=0.30):   0%|          | 0/398 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]


 Results:
  Train Loss: 3.7041
  Dev Loss:   0.2152
  Precision:  0.0000
  Recall:     0.0000
  F1 Score:   0.0000
  VA RMSE (V): inf
  VA RMSE (A): inf
  VA Variance: 0.0000
  No improvement for 1 epoch(s)

Epoch 2/15


Training (VA weight=0.30):   0%|          | 0/398 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]


 Results:
  Train Loss: 0.0018
  Dev Loss:   0.0359
  Precision:  0.8748
  Recall:     0.7249
  F1 Score:   0.7928
  VA RMSE (V): 0.8772
  VA RMSE (A): 0.8125
  VA Variance: 0.9651
  No improvement for 1 epoch(s)

Epoch 15/15


Training (VA weight=0.70):   0%|          | 0/398 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]


 Results:
  Train Loss: 0.0017
  Dev Loss:   0.0361
  Precision:  0.8791
  Recall:     0.7236
  F1 Score:   0.7938
  VA RMSE (V): 0.8739
  VA RMSE (A): 0.8116
  VA Variance: 0.9337
  No improvement for 2 epoch(s)

TRAINING COMPLETED
Best F1 score: 0.7974


## Load Best Model and Generate Predictions

In [40]:
print("Loading best model...")
model = T5ForConditionalGeneration.from_pretrained(
    f'{config.CHECKPOINT_DIR}/best_model'
).to(config.DEVICE)
tokenizer = T5Tokenizer.from_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
va_head.load_state_dict(torch.load(f'{config.CHECKPOINT_DIR}/best_va_head.pt'))
print("✓ Best model loaded")

Loading best model...
✓ Best model loaded


In [41]:
print("="*80)
print("GENERATING TEST PREDICTIONS")
print("="*80)

print("\n📝 Laptop test:")
test_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task2.jsonl')
test_laptop_dataset = DimASTETestDataset(test_laptop, tokenizer, config.MAX_INPUT_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict_with_va(model, va_head, test_laptop_loader, tokenizer, config.DEVICE)
save_predictions(predictions_laptop, test_laptop, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\n📝 Restaurant test:")
test_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task2.jsonl')
test_rest_dataset = DimASTETestDataset(test_rest, tokenizer, config.MAX_INPUT_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict_with_va(model, va_head, test_rest_loader, tokenizer, config.DEVICE)
save_predictions(predictions_rest, test_rest, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

GENERATING TEST PREDICTIONS

📝 Laptop test:
Loaded 1000 samples from eng_laptop_test_task2.jsonl
  Total triplets: 0


Predicting:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: ./improved_outputs/laptop_test_predictions.jsonl

📝 Restaurant test:
Loaded 1000 samples from eng_restaurant_test_task2.jsonl
  Total triplets: 0


Predicting:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: ./improved_outputs/restaurant_test_predictions.jsonl


## Sample Predictions

In [42]:
print("="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

print("\n🔍 Restaurant (first 5):")
for i in range(min(5, len(predictions_rest))):
    print(f"\n{i+1}. Text: {test_rest[i]['Text']}")
    print(f"   Predictions: {predictions_rest[i]}")

print("\n🔍 Laptop (first 5):")
for i in range(min(5, len(predictions_laptop))):
    print(f"\n{i+1}. Text: {test_laptop[i]['Text']}")
    print(f"   Predictions: {predictions_laptop[i]}")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)

SAMPLE PREDICTIONS

🔍 Restaurant (first 5):

1. Text: My fiance had stewed chicken and I had the curried chicken- both were excellent
   Predictions: [{'Aspect': 'both', 'Opinion': 'excellent', 'VA': '7.00#6.50'}]

2. Text: But the service , which is a huge reason to spend your hard-earned $ at a restaurant , was absolutely horrendous
   Predictions: [{'Aspect': 'service', 'Opinion': 'absolutely horrendous', 'VA': '1.57#7.70'}]

3. Text: Service was very nice but slow , especially since there were only 3 tables going the whole time we were there
   Predictions: [{'Aspect': 'Service', 'Opinion': 'very nice', 'VA': '5.77#6.42'}, {'Aspect': 'Service', 'Opinion': 'slow', 'VA': '5.77#6.42'}]

4. Text: This is a great neighborhood byob place that provides welcoming service and excellent food at a reasonable price
   Predictions: [{'Aspect': 'service', 'Opinion': 'welcoming', 'VA': '7.70#7.74'}, {'Aspect': 'food', 'Opinion': 'excellent', 'VA': '7.70#7.74'}]

5. Text: Food is good , but the ca